# Build 03-01 · Corrector inputs — targets ⨝ treatment, materialised per (version, split)

**Kernel: the analysis `.venv`** (`python3`). The one place the treatment join happens; the
corrector (03_02) and the re-evaluation (03_05) both read the artefact this notebook writes:

```
inputs/targets_<v>_<split>.parquet        (claim_id, date, observed)
   ⟕ (left join on claim_id) treatment    (decision [+ score])
   ──▶  mitigation/inputs/corrector_targets_<v>_<split>.parquet  (+ _meta.json)
```

Treatment source per version:
- **v3** — the v2 serving log's score file (`log_scores` kind, `logs/v2_score.parquet`):
  claim-level `score` + `decision` from the live model that generated v3's labels.
  Verified claim-unique on the company data (2026-09-02: `len == nunique`); a duplicate-event
  guard + collapse rule stays in §2 for reruns on refreshed exports.
- **v2** — the surviving **vehicle-status file** (§1 SOURCES): records which claims were
  fast-tracked vs sent to garage (repaired / total loss). Verified to left-join onto the v2
  training set with **zero unmatched rows** (2026-09). It carries `decision` only — the v1-era
  deciding **score is destroyed**, so v2's corrector_targets has no score column and
  rarity/pnu cannot run for v2 (thesis `tab:scheme-feasibility`); naive/transport can.

Built for **train + OOT** of each version, so 03_05 evaluates from the same artefact.

In [ ]:
# §0 — setup (analysis .venv kernel)
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config

pd.set_option("display.width", 160)
print("ROOT =", ROOT)

In [ ]:
# §1 — SOURCES / RUN_SPEC
ID_COL = "claim_id"
BUILD = {v: list(dict.fromkeys(["train", config.OOT_SPLIT[v]])) for v in ("v3", "v2")}

# v3 treatment: the v2 serving log's claim-level scores + decisions
V3_TREATMENT = config.path("log_scores", "v2")

# v2 treatment: the surviving vehicle-status file — company laptop only. Fill ALL of these in.
# NB if the file is a Z:-drive .pkl it is a JOBLIB dump (pd.read_pickle dies) — the reader below
# branches on the extension.
V2_STATUS_PATH = None                      # e.g. r"Z:/.../vehicle_status.pkl"; None -> skip v2
V2_STATUS_COLS = {"claim": "<<FILL IN>>",  # their claim-number column
                  "status": "<<FILL IN>>"} # their vehicle-status column
V2_STATUS_MAP = {
    # status value -> (decision, observed implied by the status)
    "<<fast-track total loss value>>": (1, 1),
    "<<garage repaired value>>":       (0, 0),
    "<<garage total loss value>>":     (0, 1),
}

print("build plan:", BUILD)
print("v3 treatment:", V3_TREATMENT)
print("v2 status   :", V2_STATUS_PATH or "(not set — v2 build will be skipped)")

In [ ]:
# §2 — helpers: the unique-claim gate, the collapse rule, the join+write step
def read_table(path) -> pd.DataFrame:
    """Parquet directly; a Z:-drive .pkl is a joblib dump, never pd.read_pickle."""
    p = str(path)
    if p.endswith(".parquet"):
        return pd.read_parquet(p)
    import joblib
    return joblib.load(p)


def collapse_events(df: pd.DataFrame) -> pd.DataFrame:
    """One row per claim: a decision=1 event wins (its score); else the max-score event.

    Detection is the caller's unique-count gate; this runs ONLY when duplicates exist. Keeping
    a decision=0 event for a claim that was ever fast-tracked would misfile a scrapped car as
    garage-verified, and keeping the below-tau score of a scrapped claim would contradict the
    strict rule — sorting (decision desc, score desc) and keeping the first avoids both.
    """
    out = (df.sort_values(["decision", "score"], ascending=[False, False])
             .drop_duplicates(ID_COL, keep="first"))
    print(f"  collapsed {len(df):,} event rows -> {len(out):,} claims "
          f"({len(df) - len(out):,} duplicate events dropped)")
    return out


def build_corrector_targets(version: str, split: str, treatment: pd.DataFrame,
                            source_desc: str, observed_check: pd.DataFrame | None = None) -> Path:
    """targets(split) ⟕ treatment -> corrector_targets parquet + meta. Returns the path."""
    t = pd.read_parquet(config.split_path("targets", version, split))
    m = t.merge(treatment, on=ID_COL, how="left", validate="one_to_one")
    matched = m["decision"].notna()
    n_un = int((~matched).sum())
    print(f"{version} {split}: {len(m):,} target rows | coverage {matched.mean():.1%} "
          f"({n_un:,} unmatched -> dropped)")

    n_mismatch = None
    if observed_check is not None:
        chk = m.merge(observed_check, on=ID_COL, how="left")
        both = chk["decision"].notna() & chk["observed_status"].notna()
        n_mismatch = int((chk.loc[both, "observed"].astype(int)
                          != chk.loc[both, "observed_status"].astype(int)).sum())
        if n_mismatch:
            print(f"  !! observed (targets) vs status-implied outcome disagree on {n_mismatch} rows")

    out = m.loc[matched].copy()
    out["decision"] = out["decision"].astype(int)
    p = config.split_path("corrector_targets", version, split)
    p.parent.mkdir(parents=True, exist_ok=True)
    out.to_parquet(p, index=False)
    p.with_name(p.stem + "_meta.json").write_text(json.dumps({
        "version": version, "split": split, "treatment_source": source_desc,
        "n_targets": int(len(m)), "n_unmatched_dropped": n_un,
        "n_out": int(len(out)), "has_score": bool("score" in out.columns),
        "n_scrapped": int(out["decision"].sum()),
        "n_observed_mismatch": n_mismatch,
        "columns": list(out.columns),
    }, indent=2), encoding="utf-8")
    print(f"  -> {p.name}  (cols: {list(out.columns)})")
    return p

In [ ]:
# §3 — v3: treatment from the v2 serving log (score + decision)
tr3 = read_table(V3_TREATMENT)
for c in (ID_COL, "score", "decision"):
    assert c in tr3.columns, f"log_scores is missing {c!r} (canonical names — re-run 01_export_v2_logs)"
tr3 = tr3[[ID_COL, "score", "decision"]]
n, u = len(tr3), tr3[ID_COL].nunique()
print(f"v2 log_scores: {n:,} rows / {u:,} unique {ID_COL}")
if n != u:
    tr3 = collapse_events(tr3)      # no-op gate as of 2026-09-02 (n == u on the company data)

built = []
for sp in BUILD["v3"]:
    built.append(build_corrector_targets("v3", sp, tr3, str(V3_TREATMENT)))

In [ ]:
# §4 — v2: treatment from the vehicle-status file (decision only — no v1-era score)
if V2_STATUS_PATH is None:
    print("V2_STATUS_PATH not set — v2 build skipped (fill §1 SOURCES on the company laptop)")
else:
    st = read_table(V2_STATUS_PATH)
    claim_c, status_c = V2_STATUS_COLS["claim"], V2_STATUS_COLS["status"]
    for c in (claim_c, status_c):
        assert c in st.columns, f"status file has no column {c!r} — fix V2_STATUS_COLS"
    st = st[[claim_c, status_c]].rename(columns={claim_c: ID_COL})

    unmapped = sorted(set(st[status_c].unique()) - set(V2_STATUS_MAP))
    assert not unmapped, f"unmapped status values {unmapped[:10]} — extend V2_STATUS_MAP"
    st["decision"] = st[status_c].map({k: v[0] for k, v in V2_STATUS_MAP.items()}).astype(int)
    st["observed_status"] = st[status_c].map({k: v[1] for k, v in V2_STATUS_MAP.items()}).astype(int)

    n, u = len(st), st[ID_COL].nunique()
    print(f"vehicle-status: {n:,} rows / {u:,} unique {ID_COL}")
    assert n == u, "status file is not claim-unique — decide a collapse rule before building"

    for sp in BUILD["v2"]:
        built.append(build_corrector_targets(
            "v2", sp, st[[ID_COL, "decision"]], str(V2_STATUS_PATH),
            observed_check=st[[ID_COL, "observed_status"]]))

In [ ]:
# §5 — what exists now
rows = []
for v, sps in BUILD.items():
    for sp in sps:
        p = config.split_path("corrector_targets", v, sp)
        if p.is_file():
            meta = json.loads(p.with_name(p.stem + "_meta.json").read_text(encoding="utf-8"))
            rows.append({"version": v, "split": sp, "n_out": meta["n_out"],
                         "n_scrapped": meta["n_scrapped"], "has_score": meta["has_score"],
                         "unmatched_dropped": meta["n_unmatched_dropped"],
                         "observed_mismatch": meta["n_observed_mismatch"]})
        else:
            rows.append({"version": v, "split": sp, "n_out": "(not built)"})
display(pd.DataFrame(rows).set_index(["version", "split"]))

## Notes

- Unmatched rows (targets rows the treatment source never covers) are **dropped here, once** —
  every downstream consumer then works on the same treated population.
- v2's files carry **no `score` column**; 03_02 detects that and skips rarity/pnu with a printed
  reason. v3's carry score + decision, so all four schemes run.
- Next: **03_02_reweight_mitigation.ipynb** (corrector), which now reads these files directly.